# SC4001 Phase 2, DAVIS cold-target split seeds s4002 and s4003

Accelerator must be **GPU (T4 x2 or P100)**. `src/train.py` refuses CPU by design.
Code and data come from the attached dataset `isaacwjk21/sc4001-project`; no internet or installs.

## Why this kernel exists

The cold-target split seed, meaning which genes land in which partition, moves DAVIS test MSE
further than almost anything else measured in this project. Two of the four DAVIS partitions
have been run at three training seeds each:

| split | test MSE | test CI |
| --- | --- | --- |
| `split_cold_target` (s4001) | 0.383 (0.010) | 0.792 (0.009) |
| `split_cold_target_s4004` | 0.557 (0.027) | 0.774 (0.009) |

A +0.174 MSE penalty from the partition alone, which matches the +0.18 range already measured
across KIBA's four partitions. The two untried partitions, s4002 and s4003, are the missing
part of that claim. With only the extremes run, there is no way to say whether 0.383 and 0.557
bracket the range or are simply the two draws that happened to go first, and Phase 3 is
evaluated on exactly those two partitions on the strength of them being the easiest and the
hardest. This kernel checks that.

KIBA's four partitions came out at 0.352 / 0.372 / 0.414 / 0.533, so the expectation is that
s4002 and s4003 land between 0.383 and 0.557 rather than outside them. An s4002 or s4003 that
lands outside would mean s4001 and s4004 are not the extremes, and the Phase 3 bracketing
argument would need restating.

## Design

DAVIS `split_cold_target_s4002` and `split_cold_target_s4003`, training seeds 4001-4003, six
baseline runs with `src/train.py`.

Default `patience` (15) and the default output root, deliberately. These numbers only mean
something next to the existing s4001 and s4004 figures, which were produced under the default
stopping rule, so changing it here would make the comparison incomparable. The split column is
part of the output path, so these runs cannot collide with the existing ones.

The plateau-escape failure that wrecked the Phase 3 comparison is a risk under `patience` 15,
so the summary flags any run whose best epoch is 10 or earlier. All six baseline runs on s4001
and s4004 escaped the plateau, so this is a check rather than an expectation.

## What the output answers

Per partition: each seed's test MSE, CI, best epoch and stopping epoch, then the mean and
standard deviation over the three seeds. Read against the s4001 and s4004 figures printed in
the grid cell's comment, this completes the four-way DAVIS spread and settles whether the two
partitions chosen for Phase 3 really are the ends of the range.

Expect roughly 70 minutes: 7.74 s per epoch measured for the baseline on this dataset, with
runs stopping somewhere between epoch 66 and 100.

Outputs survive only if this version is saved.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, SC4001_OUTPUT_ROOT="/kaggle/working/outputs", PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# Phase 3 needs the external kinase pool and the extra split columns, not just the Phase 2 tables.
required = ["external_kinases.csv", "external_tokens_davis.npz", "external_tokens_kiba.npz",
            "davis_extra_splits.csv", "kiba_extra_splits.csv"]
present = {p.name for p in (project / "data" / "processed").iterdir()}
missing = [name for name in required if name not in present and name + ".gz" not in present]
if missing:
    raise SystemExit("Dataset is missing Phase 3 inputs: " + str(missing))
print("phase 3 inputs OK")


In [ ]:
# Smoke the entry point on the split being run before spending quota, so a loader or
# split-resolution fault costs seconds rather than an hour.
smoke = subprocess.run(
    [sys.executable, "src/train.py", "--dataset", "davis",
     "--split", "split_cold_target_s4002", "--smoke"],
    cwd=project, env=env, check=False,
)
if smoke.returncode != 0:
    raise SystemExit("Smoke run of src/train.py failed with code " + str(smoke.returncode)
                     + "; not starting the grid.")
for stale in Path("/kaggle/working/outputs").rglob("*-smoke"):
    shutil.rmtree(stale, ignore_errors=True)
print("smoke OK")


In [ ]:
# DAVIS cold-target split seeds s4002 and s4003, filling the two partitions that have never been
# run. s4001 (MSE 0.383) and s4004 (0.557) are already measured, so this completes the four-way
# spread and says whether 0.383 and 0.557 are the ends of the range or just two draws from it.
# Default patience (15) and default output root on purpose: these must be directly comparable
# with the existing s4001 and s4004 numbers, which were produced that way. The split column is
# part of the output path, so there is no collision with those runs.
GRID = [(split, seed) for split in ("split_cold_target_s4002", "split_cold_target_s4003")
        for seed in (4001, 4002, 4003)]

started = time.time()
status = []
for split, seed in GRID:
    t0 = time.time()
    label = "baseline davis " + split + " seed" + str(seed)
    print()
    print("===== " + label + " =====", flush=True)
    run = subprocess.run(
        [sys.executable, "src/train.py", "--dataset", "davis", "--split", split,
         "--seed", str(seed), "--resume"],
        cwd=project, env=env, check=False,
    )
    elapsed = time.time() - t0
    status.append((label, run.returncode, elapsed))
    print("----- rc=" + str(run.returncode) + " wall=" + format(elapsed / 60, ".1f")
          + " min -----", flush=True)

print()
print("Grid wall time", format((time.time() - started) / 60, ".1f"), "min")
for label, rc, elapsed in status:
    print("%-56s rc=%d %6.1f min" % (label, rc, elapsed / 60))
failed = [row for row in status if row[1] != 0]
print("FAILED RUNS:", failed if failed else "none")

# The four-way DAVIS cold-target spread. Already measured elsewhere, for reference:
#   split_cold_target      (s4001)  test MSE 0.383 (sd 0.010), CI 0.792 (0.009)
#   split_cold_target_s4004          test MSE 0.557 (sd 0.027), CI 0.774 (0.009)
print()
print("DAVIS cold-target partitions measured in THIS kernel:")
import statistics
groups = {}
for path in sorted(Path("/kaggle/working/outputs").glob("**/metrics.json")):
    payload = json.loads(path.read_text())
    groups.setdefault(payload["split"], []).append(
        (payload["seed"], payload["test"]["all"]["mse"], payload["test"]["all"]["ci"],
         payload["best_epoch"], payload["stopped_epoch"]))
for split in sorted(groups):
    runs = sorted(groups[split])
    mses = [row[1] for row in runs]
    cis = [row[2] for row in runs]
    print()
    print("  %s  (%d seeds)" % (split, len(runs)))
    for seed, mse, ci, best_ep, stop_ep in runs:
        print("    seed%d  test MSE %.4f  CI %.4f  best_ep %3d  stopped_ep %3d"
              % (seed, mse, ci, best_ep, stop_ep))
    if len(runs) > 1:
        print("    mean   test MSE %.4f (sd %.4f)  CI %.4f (sd %.4f)"
              % (statistics.mean(mses), statistics.stdev(mses),
                 statistics.mean(cis), statistics.stdev(cis)))
    # A run that stops near patience after its best epoch is fine; one that stops early with a
    # best epoch in single digits is the plateau-escape failure seen in Phase 3, so flag it.
    truncated = [row for row in runs if row[3] <= 10]
    if truncated:
        print("    WARNING: %d run(s) peaked at epoch <= 10, which is the early-stopping "
              "plateau failure. Treat these as confounded." % len(truncated))


In [ ]:
# Print every metrics.json so the numbers live in the saved notebook even if files are lost.
found = sorted(Path("/kaggle/working/outputs").glob("**/metrics.json"))
print(len(found), "metrics files")
rows = []
for path in found:
    payload = json.loads(path.read_text())
    print("=" * 70)
    print(path)
    print(json.dumps(payload, indent=2))
    model = payload["test"]["all"]
    phase = "phase3" if "phase3" in str(path) else "phase2"
    rows.append((phase, payload["split"], payload["seed"], payload.get("lambda_max"),
                 model["mse"], model["ci"], payload["best_epoch"], payload["total_seconds"]))

print("=" * 100)
print("%-7s %-26s %5s %7s %9s %8s %8s %9s"
      % ("phase", "split", "seed", "lambda", "mse", "ci", "best_ep", "minutes"))
for phase, split, seed, lam, mse, ci, best_ep, secs in sorted(rows):
    lam_text = "    n/a" if lam is None else "%7.2f" % lam
    print("%-7s %-26s %5d %s %9.4f %8.4f %8d %9.1f"
          % (phase, split, seed, lam_text, mse, ci, best_ep, secs / 60))
